# 🚢 Laboratorio: Regresión Logística con el caso Titanic

## 📖 La historia
La madrugada del **15 de abril de 1912** el Titanic chocó con un iceberg. De los **891 pasajeros** de los que tenemos registro, sobrevivió el **38%**. No fue al azar: la orden fue *"mujeres y niños primero"*, y los botes salvavidas estaban más cerca de los camarotes de primera clase.

Somos analistas de una aseguradora que quiere entender el riesgo en accidentes marítimos y nos preguntan:

> ### ❓ ¿Quién sobrevivió al Titanic, y podemos predecir la supervivencia de un pasajero a partir de sus características?

Esta es la misma idea de la sesión anterior, pero con un cambio clave: **ya no predecimos un número, predecimos una categoría**. En vez de "¿cuántas millas por galón?", ahora es "¿sobrevivió: sí o no?".

## 🧭 Lo que vas a aprender
1. Por qué una regresión lineal **no sirve** para predecir una categoría, y qué hace distinto la **logística**
2. Imputar datos faltantes sin filtrar información del conjunto de prueba
3. Interpretar coeficientes como **razón de momios** (*odds ratio*)
4. Evaluar un clasificador: **matriz de confusión, precisión, recall y F1** — y por qué la exactitud engaña
5. Mover el **umbral de decisión** y leer la **curva ROC**
6. Seleccionar variables con **RFECV** y entender **cuándo hay que escalar**

## ⏱️ Agenda (3 horas)
| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación | 10 min |
| 1 | Conocer los datos | 15 min |
| 2 | Explorar: ¿quién sobrevivió? | 15 min |
| 3 | De la regresión lineal a la **logística** | 15 min |
| ☕ | Descanso | 10 min |
| 4 | Preprocesamiento: imputar y preparar variables | 20 min |
| 5 | Ajustar el modelo e **interpretarlo** | 30 min |
| 6 | Evaluar un clasificador | 25 min |
| 7 | El umbral y la curva ROC | 15 min |
| 8 | Selección automática de variables: **RFECV** | 15 min |
| 9 | Conclusiones | 10 min |

## 📋 Las variables
| Variable | Descripción |
|---|---|
| **`survived`** | 🎯 **Variable objetivo:** 1 = sobrevivió, 0 = no sobrevivió |
| `pclass` | Clase del boleto: 1, 2 o 3 |
| `sex` | `male` o `female` |
| `age` | Edad en años |
| `sibsp` | Hermanos o cónyuge a bordo |
| `parch` | Padres o hijos a bordo |
| `fare` | Tarifa pagada (libras esterlinas) |
| `embarked` | Puerto: `C` = Cherburgo, `Q` = Queenstown, `S` = Southampton |
| `adult_male`, `alone` | Columnas ya derivadas por el dataset (ojo con ellas 👀) |

---
# 0️⃣ Preparación

Ejecuta las siguientes celdas **una sola vez**. Traen las funciones que ya conoces de la sesión pasada, más las nuevas de clasificación.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_frequencies(df, columnas, top_n=None)` | Frecuencia de variables categóricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_simple_regression(x, y, results)` | Recta ajustada de un modelo OLS con 1 variable |
| `plot_actual_vs_predicted(y_real, y_pred)` | Valores reales vs predichos |
| `plot_residuals(y_real, y_pred)` | Residuales vs predichos |
| `plot_rfecv(rfecv)` | R² según el número de variables seleccionadas por RFECV |

In [ ]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas',
        labels={col: col.capitalize() for col in columns}
    )
    fig.update_layout(width=1200, height=1200, title_font_size=20)
    fig.update_traces(diagonal_visible=True)
    fig.show()


def plot_simple_regression(x, y, results):
    """Dispersión de una variable vs el objetivo con la recta ajustada por un OLS de 1 variable."""
    b0, b1 = results.params.iloc[0], results.params.iloc[1]
    x_name = getattr(x, 'name', None) or 'x'
    y_name = getattr(y, 'name', None) or 'y'
    x_line = np.linspace(np.min(x), np.max(x), 100)

    fig = px.scatter(
        x=np.asarray(x),
        y=np.asarray(y),
        opacity=0.6,
        labels={'x': x_name, 'y': y_name},
        title=f'{y_name} = {b0:.2f} + ({b1:.4f}) · {x_name}',
        template='plotly_white'
    )
    fig.add_trace(go.Scatter(
        x=x_line,
        y=b0 + b1 * x_line,
        mode='lines',
        name='Recta OLS',
        line=dict(color='red', width=3)
    ))
    fig.show()


def plot_actual_vs_predicted(y_true, y_pred, title='Real vs Predicho'):
    """Valores reales vs predichos; un modelo perfecto cae sobre la diagonal roja."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())

    fig = px.scatter(
        x=y_true,
        y=y_pred,
        opacity=0.5,
        labels={'x': 'Valor real', 'y': 'Valor predicho'},
        title=title,
        template='plotly_white'
    )
    fig.add_shape(
        type='line', x0=lo, y0=lo, x1=hi, y1=hi,
        line=dict(color='red', dash='dash')
    )
    fig.show()


def plot_residuals(y_true, y_pred, title='Residuales vs Predicho'):
    """Residuales vs predichos; buscamos una nube sin patrón alrededor de 0."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)

    fig = px.scatter(
        x=y_pred,
        y=y_true - y_pred,
        opacity=0.5,
        labels={'x': 'Valor predicho', 'y': 'Residual (real − predicho)'},
        title=title,
        template='plotly_white'
    )
    fig.add_hline(y=0, line_dash='dash', line_color='red')
    fig.show()


def plot_rfecv(rfecv):
    """R² promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name='R² promedio (CV)'
    ))
    fig.update_layout(
        title='RFECV — R² según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title='R² (validación cruzada)',
        template='plotly_white',
        width=900, height=450
    )
    fig.show()

### 🧰 Funciones de clasificación

Ejecuta la celda una vez; después sólo llamas a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `ajustar_logit(X_train, y_train)` | Ajusta una regresión logística (ya agrega la constante) |
| `predecir_proba(results, X)` | Probabilidad estimada de la clase positiva |
| `razon_de_momios(results)` | Coeficientes, razón de momios y p-value en una tabla |
| `evaluar_clasificador(nombre, y_real, y_pred)` | Exactitud, precisión, recall y F1 |
| `plot_tasa_por_categoria(df, columna, objetivo)` | Tasa de la clase positiva por categoría |
| `plot_matriz_confusion(y_real, y_pred, etiquetas)` | Matriz de confusión |
| `plot_curva_roc(y_real, y_proba)` | Curva ROC y AUC |
| `plot_metricas_por_umbral(y_real, y_proba)` | Cómo cambian las métricas al mover el umbral |
| `plot_sigmoide()` | La curva que convierte cualquier número en probabilidad |

In [ ]:
# Funciones de clasificación: ajustar, evaluar y diagnosticar modelos de clasificación
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)


def ajustar_logit(X_train, y_train):
    """Ajusta una regresión logística (agrega la constante automáticamente)."""
    return sm.Logit(y_train, sm.add_constant(X_train)).fit(disp=0)


def predecir_proba(results, X):
    """Probabilidad estimada de la clase positiva, entre 0 y 1."""
    return results.predict(sm.add_constant(X))


def razon_de_momios(results):
    """Coeficientes en log-odds, su razón de momios (odds ratio) y el p-value."""
    return pd.DataFrame({
        'coef (log-odds)': results.params.round(4),
        'razón de momios': np.exp(results.params).round(3),
        'p-value': results.pvalues.round(4),
    })


def evaluar_clasificador(nombre, y_true, y_pred):
    """Imprime y regresa exactitud, precisión, recall y F1."""
    metricas = {
        'modelo': nombre,
        'exactitud': round(accuracy_score(y_true, y_pred), 3),
        'precisión': round(precision_score(y_true, y_pred, zero_division=0), 3),
        'recall': round(recall_score(y_true, y_pred, zero_division=0), 3),
        'F1': round(f1_score(y_true, y_pred, zero_division=0), 3),
    }
    print(f"{nombre}: exactitud {metricas['exactitud']} | precisión {metricas['precisión']} "
          f"| recall {metricas['recall']} | F1 {metricas['F1']}")
    return metricas


def plot_tasa_por_categoria(df, columna, objetivo):
    """Proporción de la clase positiva dentro de cada categoría."""
    tasa = df.groupby(columna, as_index=False)[objetivo].mean()

    fig = px.bar(
        tasa,
        x=columna,
        y=objetivo,
        text_auto='.1%',
        title=f'Tasa de {objetivo} por {columna}',
        template='plotly_white'
    )
    fig.update_layout(yaxis_tickformat='.0%', yaxis_title=f'Tasa de {objetivo}', width=700, height=450)
    fig.update_xaxes(type='category')
    fig.show()


def plot_matriz_confusion(y_true, y_pred, etiquetas=('No', 'Sí'), title='Matriz de confusión'):
    """Aciertos y errores del clasificador: la diagonal son los aciertos."""
    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred),
        index=[f'Real: {e}' for e in etiquetas],
        columns=[f'Predicho: {e}' for e in etiquetas]
    )

    fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues', title=title)
    fig.update_layout(width=650, height=450)
    fig.show()


def plot_curva_roc(y_true, y_proba, title='Curva ROC'):
    """Compromiso entre verdaderos y falsos positivos al mover el umbral."""
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines', name=f'Modelo (AUC = {auc:.3f})',
        line=dict(color='steelblue', width=3)
    ))
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1], mode='lines', name='Azar (AUC = 0.5)',
        line=dict(color='red', dash='dash')
    ))
    fig.update_layout(
        title=title,
        xaxis_title='Falsos positivos (1 − especificidad)',
        yaxis_title='Verdaderos positivos (recall)',
        template='plotly_white', width=700, height=550
    )
    fig.show()


def plot_metricas_por_umbral(y_true, y_proba):
    """Cómo cambian exactitud, precisión, recall y F1 según el umbral de decisión."""
    umbrales = np.arange(0.05, 0.96, 0.05)
    filas = []
    for u in umbrales:
        y_pred = (y_proba >= u).astype(int)
        filas.append({
            'umbral': round(u, 2),
            'exactitud': accuracy_score(y_true, y_pred),
            'precisión': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'F1': f1_score(y_true, y_pred, zero_division=0),
        })

    datos = pd.DataFrame(filas).melt(id_vars='umbral', var_name='métrica', value_name='valor')

    fig = px.line(
        datos, x='umbral', y='valor', color='métrica', markers=True,
        title='Métricas según el umbral de decisión', template='plotly_white'
    )
    fig.add_vline(x=0.5, line_dash='dash', line_color='gray', annotation_text='umbral por defecto')
    fig.update_layout(width=900, height=500)
    fig.show()


def plot_rfecv_clasificacion(rfecv, metrica='exactitud'):
    """Desempeño promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name=f'{metrica} promedio (CV)'
    ))
    fig.add_vline(x=rfecv.n_features_, line_dash='dash', line_color='red',
                  annotation_text=f'elegido: {rfecv.n_features_}')
    fig.update_layout(
        title='RFECV — desempeño según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title=f'{metrica.capitalize()} (validación cruzada)',
        template='plotly_white', width=900, height=450
    )
    fig.show()


def plot_sigmoide():
    """La función que convierte cualquier número en una probabilidad entre 0 y 1."""
    z = np.linspace(-8, 8, 200)

    fig = px.line(
        x=z, y=1 / (1 + np.exp(-z)),
        labels={'x': 'z = β₀ + β₁x₁ + β₂x₂ + …', 'y': 'Probabilidad estimada'},
        title='Función sigmoide (logística)', template='plotly_white'
    )
    fig.add_hline(y=0.5, line_dash='dash', line_color='red', annotation_text='umbral 0.5')
    fig.add_vline(x=0, line_dash='dot', line_color='gray')
    fig.update_traces(line=dict(color='steelblue', width=3))
    fig.update_layout(width=800, height=450)
    fig.show()

### 📥 Cargar los datos
Misma fuente que la sesión anterior: una base **SQLite** que descargamos y consultamos con SQL.

In [ ]:
import requests, sqlite3, pandas as pd

url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/titanic.db"
r = requests.get(url)

with open("titanic.db", "wb") as f:
    f.write(r.content)

conn = sqlite3.connect("titanic.db")

query = """
SELECT
    O.survived,
    O.pclass,
    O.age,
    O.sibsp,
    O.parch,
    O.fare,
    O.adult_male,
    O.alone,
    S.sex,
    E.embarked
FROM
    Observation AS O
JOIN
    Sex AS S ON O.sex_id = S.sex_id
JOIN
    Embarked AS E ON O.embarked_id = E.embarked_id
"""

df = pd.read_sql_query(query, conn)
df.head()

,survived,pclass,age,sibsp,parch,fare,adult_male,alone,sex,embarked
0,0,3,22.0,1,0,7.2500,1,0,male,S
1,1,1,38.0,1,0,71.2833,0,0,female,C
2,1,3,26.0,0,0,7.9250,0,1,female,S
3,1,1,35.0,1,0,53.1000,0,0,female,S
4,0,3,35.0,0,0,8.0500,1,1,male,S


---
# 1️⃣ Conocer los datos (15 min)

Tres preguntas de siempre: ¿cuántos datos hay?, ¿faltan valores?, ¿cómo está repartida la variable objetivo?

In [ ]:
# ¿Cuántas filas y columnas? ¿De qué tipo es cada columna?
# Pista: df.shape y df.dtypes
# ✍️ Tu código aquí
print(df.shape)
df.dtypes

(891, 10)


,0
survived,int64
pclass,int64
age,float64
sibsp,int64
parch,int64
fare,float64
adult_male,int64
alone,int64
sex,object
embarked,object


In [ ]:
# ¿Hay valores nulos y en qué columnas?
# Pista: df.isnull().sum()
# ✍️ Tu código aquí
df.isnull().sum()

,0
survived,0
pclass,0
age,177
sibsp,0
parch,0
fare,0
adult_male,0
alone,0
sex,0
embarked,2


### Aquí **no** podemos eliminar las filas con nulos

En la sesión pasada eliminamos los nulos de `horsepower` porque eran el **1.5%** de las filas. Calcula el porcentaje aquí antes de decidir.

In [ ]:
# ¿Qué porcentaje de filas tiene nulos en 'age' y en 'embarked'?
# Pista: df[['age', 'embarked']].isnull().mean()
# ✍️ Tu código aquí
df[['age', 'embarked']].isnull().mean()

,0
age,0.198653
embarked,0.002245


✍️ **¿Qué observas?**
- ¿Qué porcentaje de nulos tiene `age`? ¿Te parece razonable eliminar esas filas?
- ¿Y `embarked`?

- **age** tiene **19.87%** de nulos (177 pasajeros). No conviene eliminarlos porque perderíamos casi la quinta parte de los datos, que es mucho más del 5% de la regla. Mejor los imputamos.
- **embarked** solo tiene **0.22%** de nulos (2 pasajeros). Se podrían borrar sin problema, pero como son tan pocos los rellenamos con el puerto más común.

### ¿Cómo está repartida la variable objetivo?
En clasificación esto es lo primero que se revisa: si el 99% de los casos fueran de una sola clase, un modelo que siempre responda esa clase acertaría el 99% de las veces sin aprender nada.

In [ ]:
# ¿Qué proporción de pasajeros sobrevivió?
# Pista: df['survived'].value_counts(normalize=True)
# ✍️ Tu código aquí
df['survived'].value_counts(normalize=True)

,proportion
survived,
0,0.616162
1,0.383838


✍️ ¿Qué proporción sobrevivió? Anótala: la usaremos como punto de comparación más adelante.

Sobrevivió el **38.4%** de los pasajeros y murió el **61.6%**. Las clases no están perfectamente balanceadas, pero tampoco es un caso extremo. Ojo: un modelo que diga que todos murieron ya le atinaría al 61.6%, así que nuestro modelo tiene que ganarle a ese número.

---
# 2️⃣ Explorar: ¿quién sobrevivió? (15 min)

Con una variable objetivo de 0 y 1, el gráfico más útil es la **tasa de supervivencia por grupo**: el promedio de `survived` dentro de cada categoría.

In [ ]:
# Tasa de supervivencia por sexo
# Llama a plot_tasa_por_categoria(df, 'sex', 'survived')
# ✍️ Tu código aquí
plot_tasa_por_categoria(df, 'sex', 'survived')

In [ ]:
# Tasa de supervivencia por clase
# ✍️ Tu código aquí
plot_tasa_por_categoria(df, 'pclass', 'survived')

In [ ]:
# Tasa de supervivencia por puerto de embarque
# ✍️ Tu código aquí
plot_tasa_por_categoria(df, 'embarked', 'survived')

# para revisar que clase viajaba en cada puerto
pd.crosstab(df['embarked'], df['pclass'], normalize='index').round(3)

pclass,1,2,3
embarked,,,
C,0.506,0.101,0.393
Q,0.026,0.039,0.935
S,0.197,0.255,0.548


### ¿Y la edad?
`age` es numérica, así que la agrupamos en rangos para poder ver la tasa por grupo.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
df_edad = df.assign(
    grupo_edad=pd.cut(df['age'], [0, 16, 30, 50, 100], labels=['0-16', '17-30', '31-50', '51+'])
)

plot_tasa_por_categoria(df_edad, 'grupo_edad', 'survived')

/tmp/ipykernel_4518/4269669596.py:46: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



✍️ **¿Qué observas?**
- ¿Qué variable parece influir más en la supervivencia?
- ¿Qué grupo de edad sobrevivió más?
- Cherburgo tiene mejor tasa que Southampton. ¿Crees que el puerto salva vidas, o hay otra explicación? (Pista: revisa qué clase viajaba en cada puerto con `pd.crosstab(df['embarked'], df['pclass'], normalize='index')`)

- La variable que más influye es el **sexo**: sobrevivió el **74.2%** de las mujeres contra solo el **18.9%** de los hombres. La **clase** también pesa mucho: 63% en primera, 47.3% en segunda y 24.2% en tercera.
- El grupo de edad que más sobrevivió fue el de **0 a 16 años**, con 55%, lo que cuadra con "mujeres y niños primero".
- No creo que el puerto salve vidas. Con el crosstab se ve que en **Cherburgo el 50.6% viajaba en primera clase**, contra solo 19.7% en Southampton. Y en Queenstown el 93.5% era de tercera. O sea, el puerto refleja la **clase** de los pasajeros, que es lo que realmente importa.

---
# 3️⃣ De la regresión lineal a la logística (15 min)

### ¿Por qué no usar la regresión lineal de la sesión pasada?

Imagina ajustar una recta a `survived` (que sólo vale 0 o 1) usando la edad:

$$\text{survived} = \beta_0 + \beta_1 \cdot \text{age}$$

Una recta **no tiene límites**: para edades altas predeciría valores negativos, y para otras, valores mayores a 1. **¿Qué significa una supervivencia de -0.3 o de 1.4?** Nada.

### La solución: la función sigmoide

La regresión logística toma la misma combinación lineal de siempre…

$$z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots$$

…y la pasa por una función que **aplasta cualquier número al rango de 0 a 1**:

$$P(\text{sobrevivir}) = \frac{1}{1 + e^{-z}}$$

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
plot_sigmoide()

✅ **Qué observar en la curva:**
- Si $z = 0$, la probabilidad es exactamente **0.5**
- Valores muy negativos de $z$ → probabilidad cercana a **0**; muy positivos → cercana a **1**
- Nunca se sale del rango 0 a 1, sin importar qué tan grande sea $z$

### Lo que cambia y lo que no

| | Regresión lineal | Regresión logística |
|---|---|---|
| Predice | Un número cualquiera | Una **probabilidad** (0 a 1) |
| Ecuación | $y = \beta_0 + \beta_1x$ | $P = \dfrac{1}{1+e^{-(\beta_0 + \beta_1x)}}$ |
| Coeficientes | Cambio en $y$ por unidad de $x$ | Cambio en el **log-odds** por unidad de $x$ |
| Se ajusta con | Mínimos cuadrados (OLS) | Máxima verosimilitud |
| Métricas | R², RMSE | Exactitud, precisión, recall, F1 |

Lo que **no** cambia: train/test, el papel de los p-values, y el cuidado con las variables redundantes.

---
# ☕ Descanso (10 min)

---
# 4️⃣ Preprocesamiento (20 min)

### Primero dividir, después imputar
El orden importa. Si imputamos la edad con la mediana de **todos** los datos y luego dividimos, la mediana del conjunto de prueba se habrá colado en el entrenamiento. Eso se llama **fuga de datos** (*data leakage*) y hace que el modelo se vea mejor de lo que es.

Usamos `stratify=y` para que train y test conserven la misma proporción de sobrevivientes.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. X = df sin 'survived' y sin las columnas derivadas 'adult_male' y 'alone'
# 2. y = la columna 'survived'
# 3. train_test_split 80/20 con random_state=42 y stratify=y
# 4. Imprime las formas y la tasa de supervivencia de cada conjunto
# ✍️ Tu código aquí
X = df.drop(columns=['survived', 'adult_male', 'alone'])
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print("Tasa de supervivencia train:", round(y_train.mean(), 3))
print("Tasa de supervivencia test:", round(y_test.mean(), 3))

(712, 7) (179, 7)
Tasa de supervivencia train: 0.383
Tasa de supervivencia test: 0.385


### Imputar los faltantes
- `age` → la **mediana** (es robusta a valores extremos)
- `embarked` → la **moda**, es decir, el puerto más frecuente

Ambos valores se calculan **sólo con train** y se aplican a los dos conjuntos.

> ⚠️ **Recuerda (pandas 3):** `df['col'].fillna(valor, inplace=True)` ya no modifica nada. Usa `df['col'] = df['col'].fillna(valor)`.

In [ ]:
# 1. Calcula mediana_edad y moda_puerto SOLO con X_train
#    Pistas: X_train['age'].median()  y  X_train['embarked'].mode()[0]
# 2. Copia X_train y X_test en X_train_prep y X_test_prep
# 3. Rellena 'age' y 'embarked' en AMBOS con esos dos valores de train
# 4. Verifica que ya no queden nulos
# ✍️ Tu código aquí
mediana_edad = X_train['age'].median()
moda_puerto = X_train['embarked'].mode()[0]
print("Mediana de edad (train):", mediana_edad)
print("Moda de puerto (train):", moda_puerto)

X_train_prep = X_train.copy()
X_test_prep = X_test.copy()

for datos in (X_train_prep, X_test_prep):
    datos['age'] = datos['age'].fillna(mediana_edad)
    datos['embarked'] = datos['embarked'].fillna(moda_puerto)

print("Nulos en train:", X_train_prep.isnull().sum().sum())
print("Nulos en test:", X_test_prep.isnull().sum().sum())

Mediana de edad (train): 28.5
Moda de puerto (train): S
Nulos en train: 0
Nulos en test: 0


### Crear variables nuevas (*feature engineering*)
A veces una variable útil no viene en los datos, pero se puede construir:
- **`viaja_solo`**: 1 si el pasajero no lleva familiares a bordo (`sibsp + parch == 0`)
- **`es_menor`**: 1 si tiene 16 años o menos

`np.where(condición, valor_si_verdadero, valor_si_falso)` es la forma directa de hacerlo.

In [ ]:
import numpy as np

# En X_train_prep y X_test_prep:
# 1. viaja_solo = 1 si (sibsp + parch) == 0, si no 0   → np.where(...)
# 2. es_menor   = 1 si age <= 16, si no 0
# 3. Elimina las columnas 'sibsp' y 'parch' (ya están resumidas en viaja_solo)
# ✍️ Tu código aquí
for datos in (X_train_prep, X_test_prep):
    datos['viaja_solo'] = np.where(datos['sibsp'] + datos['parch'] == 0, 1, 0)
    datos['es_menor'] = np.where(datos['age'] <= 16, 1, 0)

X_train_prep = X_train_prep.drop(columns=['sibsp', 'parch'])
X_test_prep = X_test_prep.drop(columns=['sibsp', 'parch'])

X_train_prep.head()

,pclass,age,fare,sex,embarked,viaja_solo,es_menor
692,3,28.5,56.4958,male,S,1,0
481,2,28.5,0.0000,male,S,1,0
527,1,28.5,221.7792,male,S,1,0
855,3,18.0,9.3500,female,S,0,0
801,2,31.0,26.2500,female,S,0,0


> 💡 **¿Y el escalado?** Aquí no escalamos nada: la regresión logística no lo necesita para ajustarse ni para predecir. En la Parte 8 verás cuándo sí hace falta.

### Variables categóricas a dummies
Igual que con `origin` en la sesión pasada: una columna menos que categorías, y la categoría que quitamos se vuelve la **referencia** contra la que se comparan las demás.

Nuestras referencias serán: **3ª clase**, puerto **Southampton** y **mujer**. Así cada coeficiente se leerá como *"comparado con una mujer de tercera clase que embarcó en Southampton"*.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.preprocessing import OneHotEncoder

categoricas = ['pclass', 'embarked', 'sex']

ohe = OneHotEncoder(drop=[3, 'S', 'female'], sparse_output=False)

dummies_train = pd.DataFrame(
    ohe.fit_transform(X_train_prep[categoricas]),
    columns=ohe.get_feature_names_out(),
    index=X_train_prep.index
)
dummies_test = pd.DataFrame(
    ohe.transform(X_test_prep[categoricas]),
    columns=ohe.get_feature_names_out(),
    index=X_test_prep.index
)

X_train_enc = pd.concat([X_train_prep.drop(columns=categoricas), dummies_train], axis=1)
X_test_enc = pd.concat([X_test_prep.drop(columns=categoricas), dummies_test], axis=1)

X_train_enc.head()

,age,fare,viaja_solo,es_menor,pclass_1,pclass_2,embarked_C,embarked_Q,sex_male
692,28.5,56.4958,1,0,0.0,0.0,0.0,0.0,1.0
481,28.5,0.0000,1,0,0.0,1.0,0.0,0.0,1.0
527,28.5,221.7792,1,0,1.0,0.0,0.0,0.0,1.0
855,18.0,9.3500,0,0,0.0,0.0,0.0,0.0,0.0
801,31.0,26.2500,0,0,0.0,1.0,0.0,0.0,0.0


---
# 5️⃣ Ajustar el modelo e interpretarlo (30 min)

Usamos `statsmodels`, igual que en la sesión pasada, para tener el resumen estadístico completo.

In [ ]:
# 1. modelo_completo = ajustar_logit(X_train_enc, y_train)
# 2. print(modelo_completo.summary())
# ✍️ Tu código aquí
modelo_completo = ajustar_logit(X_train_enc, y_train)
print(modelo_completo.summary())

                           Logit Regression Results                           
Dep. Variable:               survived   No. Observations:                  712
Model:                          Logit   Df Residuals:                      702
Method:                           MLE   Df Model:                            9
Date:                Mon, 26 Oct 2026   Pseudo R-squ.:                  0.3415
Time:                        04:26:27   Log-Likelihood:                -312.14
converged:                       True   LL-Null:                       -473.99
Covariance Type:            nonrobust   LLR p-value:                 2.422e-64
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.5506      0.384      1.432      0.152      -0.203       1.304
age           -0.0250      0.010     -2.450      0.014      -0.045      -0.005
fare           0.0008      0.003      0.297      0.7

### 🔍 Cómo leer este resumen
Es casi igual al de la sesión pasada, con dos diferencias:

| Dónde | Qué es |
|---|---|
| `Pseudo R-squ.` | Parecido al R², pero no se interpreta como "% explicado". Sirve para comparar modelos del mismo tipo |
| Columna `coef` | El cambio en el **log-odds**, no en la probabilidad. Por eso los traducimos abajo |
| Columna `P>\|z\|` | El p-value de siempre: < 0.05 → la variable es significativa |

### De log-odds a razón de momios
Un coeficiente de `-2.62` no dice gran cosa. Pero si le aplicamos $e^{-2.62} = 0.073$, ya se puede leer: **los momios de sobrevivir se multiplican por 0.073**, o sea que caen un 93%.

**Momios** (*odds*) = probabilidad de que pase entre probabilidad de que no pase. Si 3 de cada 4 sobreviven, los momios son 3 a 1.

In [ ]:
# Traduce los coeficientes a razón de momios
# Pista: razon_de_momios(modelo_completo)
# ✍️ Tu código aquí
# para la pregunta de fare: correlacion entre fare y pclass
print(df[['fare', 'pclass']].corr().round(2))

razon_de_momios(modelo_completo)

        fare  pclass
fare    1.00   -0.55
pclass -0.55    1.00


,coef (log-odds),razón de momios,p-value
const,0.5506,1.734,0.1521
age,-0.0250,0.975,0.0143
fare,0.0008,1.001,0.7668
viaja_solo,0.1536,1.166,0.5079
es_menor,0.7075,2.029,0.0837
pclass_1,2.3408,10.390,0.0000
pclass_2,1.3363,3.805,0.0000
embarked_C,0.4978,1.645,0.0617
embarked_Q,0.8280,2.289,0.0286
sex_male,-2.6176,0.073,0.0000


✍️ **Interpreta tus resultados:**
- ¿Cuál es la variable con el coeficiente más fuerte? Tradúcela a momios en una frase.
- ¿Cuánto multiplica los momios viajar en 1ª clase frente a 3ª?
- ¿Qué variables tienen p-value mayor a 0.05? ¿Se te ocurre por qué `fare` no aporta, si pagar más parecería ayudar? (Pista: `df[['fare', 'pclass']].corr()`)

- La variable con el coeficiente más fuerte es **sex_male** (-2.6176). Su razón de momios es **0.073**: ser hombre multiplica los momios de sobrevivir por 0.073, o sea que bajan como **93%** comparado con una mujer.
- Viajar en **1ª clase** multiplica los momios de sobrevivir por **10.39** frente a 3ª clase. En 2ª clase se multiplican por 3.8.
- Tienen p-value mayor a 0.05: **fare (0.767), viaja_solo (0.508), es_menor (0.084) y embarked_C (0.062)**. Creo que **fare** no aporta porque está correlacionada con **pclass** (-0.55): los que pagaban más eran justo los de primera clase, así que esa información ya la trae pclass y fare no agrega nada nuevo.

### 👀 La trampa que dejamos pendiente

Al inicio quitamos `alone` y `adult_male` porque el dataset ya las traía derivadas. Resulta que **`alone` es exactamente la misma columna que la `viaja_solo` que acabas de construir**. Veamos qué pasa si el modelo recibe las dos.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
alone_original = np.where(X_train['sibsp'] + X_train['parch'] > 0, 0, 1)
print("¿'alone' y 'viaja_solo' son idénticas?",
      (alone_original == X_train_enc['viaja_solo']).all())

X_con_duplicada = X_train_enc.assign(alone=alone_original)

try:
    ajustar_logit(X_con_duplicada, y_train)
except Exception as error:
    print(f"\n💥 {type(error).__name__}: {error}")

¿'alone' y 'viaja_solo' son idénticas? True


✍️ ¿Qué error apareció? Conéctalo con lo que viste de multicolinealidad y VIF en la sesión pasada: ¿en qué se parecen?

En mi caso **no salió un error**: el modelo sí se ajustó, pero hizo algo raro. El coeficiente que antes tenía viaja_solo (0.1536) se **repartió a la mitad** entre viaja_solo y alone (0.0768 cada una), y sus errores estándar y p-values salen vacíos (NaN). O sea, el modelo no puede saber a cuál de las dos darle el crédito.

Es lo mismo que la multicolinealidad de la sesión pasada, pero llevado al extremo: si dos variables son **exactamente iguales**, una se explica al 100% con la otra y su **VIF sería infinito**. En MPG pasaba con weight, displacement y cylinders, que contaban casi la misma historia y daban signos raros y p-values altos.

---
# 6️⃣ Evaluar un clasificador (25 min)

El modelo devuelve **probabilidades**. Para tomar una decisión hay que fijar un **umbral**: por defecto, probabilidad ≥ 0.5 → predecimos "sobrevivió".

In [ ]:
# 1. proba_test = predecir_proba(modelo_completo, X_test_enc)
# 2. pred_test = (proba_test >= 0.5).astype(int)
# 3. Muestra las primeras probabilidades con .head().round(3)
# ✍️ Tu código aquí
proba_test = predecir_proba(modelo_completo, X_test_enc)
pred_test = (proba_test >= 0.5).astype(int)
proba_test.head().round(3)

,0
565,0.066
160,0.041
553,0.123
860,0.044
241,0.663


In [ ]:
# Guardaremos las métricas de cada modelo para compararlos al final
comparacion = []

# Pista: comparacion.append(evaluar_clasificador('Logística (umbral 0.5)', y_test, pred_test))
# ✍️ Tu código aquí
comparacion.append(evaluar_clasificador('Logística (umbral 0.5)', y_test, pred_test))

Logística (umbral 0.5): exactitud 0.771 | precisión 0.741 | recall 0.623 | F1 0.677


### La matriz de confusión
Cuatro casillas: los aciertos están en la diagonal.

| | Predicho: No | Predicho: Sí |
|---|---|---|
| **Real: No** | Verdadero negativo ✅ | Falso positivo ❌ |
| **Real: Sí** | Falso negativo ❌ | Verdadero positivo ✅ |

In [ ]:
# Llama a plot_matriz_confusion(y_test, pred_test, etiquetas=('No sobrevivió', 'Sobrevivió'))
# ✍️ Tu código aquí
plot_matriz_confusion(y_test, pred_test, etiquetas=('No sobrevivió', 'Sobrevivió'))
print(confusion_matrix(y_test, pred_test))

[[95 15]
 [26 43]]


✍️ **Lee tu matriz de confusión:**
- ¿Cuántos pasajeros clasificó bien en total?
- ¿Cuántos falsos negativos hay (sobrevivieron y el modelo dijo que no)?
- Con tus números, explica con tus palabras qué significan la precisión y el recall.

- Clasificó bien **138 de 179** pasajeros (95 que no sobrevivieron y 43 que sí), o sea el 77.1%.
- Hay **26 falsos negativos**: pasajeros que sí sobrevivieron y el modelo dijo que no.
- **Precisión (0.741):** de los 58 pasajeros que el modelo dijo que sobrevivían, 43 de verdad sobrevivieron. Es qué tanto le atina cuando dice "sí".
- **Recall (0.623):** de los 69 que en realidad sobrevivieron, el modelo encontró 43. Es qué tantos sobrevivientes logra detectar.

### ¿Contra qué comparamos? Dos modelos tontos

Un número suelto no dice si el modelo es bueno. Necesitamos puntos de comparación:
1. **"Todos mueren":** predecir siempre 0, sin mirar nada
2. **"Mujeres y niños primero":** la regla histórica, en su versión más simple — *sobreviven las mujeres*

In [ ]:
# 1. pred_todos_mueren = np.zeros(len(y_test), dtype=int)
# 2. pred_regla_mujeres = (X_test['sex'] == 'female').astype(int)
# 3. Evalúa las dos con evaluar_clasificador() y agrégalas a comparacion
# ✍️ Tu código aquí
pred_todos_mueren = np.zeros(len(y_test), dtype=int)
pred_regla_mujeres = (X_test['sex'] == 'female').astype(int)

comparacion.append(evaluar_clasificador('Todos mueren', y_test, pred_todos_mueren))
comparacion.append(evaluar_clasificador('Regla: sobreviven mujeres', y_test, pred_regla_mujeres))

Todos mueren: exactitud 0.615 | precisión 0.0 | recall 0.0 | F1 0.0
Regla: sobreviven mujeres: exactitud 0.777 | precisión 0.738 | recall 0.652 | F1 0.692


✍️ **Compara los tres resultados:**
- ¿Qué exactitud logra "todos mueren"? ¿Cuál es su recall? ¿Qué te dice eso sobre confiar sólo en la exactitud?
- ¿Le ganó la regla simple a tu modelo? 🤔 ¿Para qué sirve entonces el modelo? Anota tu hipótesis y la revisamos en la Parte 7.

- "Todos mueren" tiene una exactitud de **0.615**, que suena bien, pero su **recall es 0**: no detecta ni a un solo sobreviviente. Eso demuestra que la exactitud sola engaña cuando las clases no están parejas.
- Sí, la regla de "sobreviven las mujeres" le ganó por poquito a mi modelo con umbral 0.5: exactitud 0.777 contra 0.771, y F1 0.692 contra 0.677. Mi hipótesis es que el modelo sirve porque no da solo un sí o no, sino una **probabilidad**, y eso permite ajustar el umbral según lo que más nos importe.

---
# 7️⃣ El umbral y la curva ROC (15 min)

La regla simple da un **sí o no** y ahí se acaba. El modelo da una **probabilidad**, y eso permite algo que la regla no puede: **elegir qué error prefieres cometer**.

El 0.5 no es sagrado. Bajarlo hace al modelo más "optimista" (detecta más sobrevivientes, pero se equivoca más); subirlo lo vuelve más exigente.

In [ ]:
# ¿Cómo cambian las métricas al mover el umbral?
# Llama a plot_metricas_por_umbral(y_test, proba_test)
# ✍️ Tu código aquí
plot_metricas_por_umbral(y_test, proba_test)

In [ ]:
# Prueba un umbral distinto (por ejemplo 0.4), evalúalo y agrégalo a comparacion
# Pista: pred_umbral_04 = (proba_test >= 0.4).astype(int)
# ✍️ Tu código aquí
pred_umbral_04 = (proba_test >= 0.4).astype(int)
comparacion.append(evaluar_clasificador('Logística (umbral 0.4)', y_test, pred_umbral_04))

# probamos varios umbrales para ver cual da mejor F1
for u in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
    p = (proba_test >= u).astype(int)
    print(u, "F1:", round(f1_score(y_test, p), 3), "| recall:", round(recall_score(y_test, p), 3), "| precision:", round(precision_score(y_test, p), 3))

Logística (umbral 0.4): exactitud 0.782 | precisión 0.697 | recall 0.768 | F1 0.731
0.3 F1: 0.709 | recall: 0.812 | precision: 0.629
0.35 F1: 0.724 | recall: 0.797 | precision: 0.663
0.4 F1: 0.731 | recall: 0.768 | precision: 0.697
0.45 F1: 0.696 | recall: 0.681 | precision: 0.712
0.5 F1: 0.677 | recall: 0.623 | precision: 0.741
0.55 F1: 0.683 | recall: 0.594 | precision: 0.804
0.6 F1: 0.672 | recall: 0.565 | precision: 0.83


✍️ Prueba varios umbrales. ¿Con cuál obtienes el mejor F1? ¿Qué le pasa al recall cuando bajas el umbral, y por qué?

Probé umbrales de 0.3 a 0.6, y el mejor F1 fue con **0.4 (F1 = 0.731)**. Ahí la exactitud también sube a 0.782.

Cuando bajo el umbral, el **recall sube** (con 0.3 llega a 0.812), porque el modelo necesita menos probabilidad para decir que alguien sobrevivió y detecta a más sobrevivientes. El costo es que la **precisión baja** (0.629 con 0.3), porque también se equivoca más diciendo "sí" a gente que no sobrevivió.

### La curva ROC: el modelo sin comprometerse con un umbral
La curva ROC recorre **todos los umbrales posibles** y grafica cuántos sobrevivientes detectas (eje Y) contra cuántas falsas alarmas generas (eje X).

El **AUC** (área bajo la curva) resume esa curva en un número: la probabilidad de que, tomando un sobreviviente y un no sobreviviente al azar, el modelo le asigne mayor probabilidad al que sí sobrevivió. 0.5 es azar puro, 1.0 es perfecto.

In [ ]:
# Llama a plot_curva_roc(y_test, proba_test)
# ✍️ Tu código aquí
plot_curva_roc(y_test, proba_test)
print("AUC:", round(roc_auc_score(y_test, proba_test), 3))

AUC: 0.837


✍️ ¿Qué AUC obtuviste? Con eso en mano, responde la pregunta que dejaste pendiente: ¿qué te da el modelo que la regla simple no puede dar?

Obtuve un **AUC de 0.837**, bastante arriba del 0.5 del azar.

Eso confirma mi hipótesis: la regla simple solo da un sí o no y no se puede ajustar, mientras que el modelo da una **probabilidad** para cada pasajero y ordena bien quién tiene más chance de sobrevivir. Con eso puedo mover el umbral según lo que necesite. Por ejemplo, con 0.4 el modelo ya le gana a la regla de las mujeres (F1 0.731 contra 0.692 y exactitud 0.782 contra 0.777).

---
# 7️⃣.5 Depurar por p-values

Antes de cerrar, quitamos las dos variables que no eran significativas (`fare` y `viaja_solo`), como hicimos en la sesión pasada con los p-values.

In [ ]:
# 1. columnas_reducido = X_train_enc.columns.drop([...])  ← las dos variables no significativas
# 2. Ajusta modelo_reducido con esas columnas
# 3. Predice, aplica el umbral 0.5 y evalúa con evaluar_clasificador()
# 4. Muestra su razon_de_momios()
# ✍️ Tu código aquí
columnas_reducido = X_train_enc.columns.drop(['fare', 'viaja_solo'])
modelo_reducido = ajustar_logit(X_train_enc[columnas_reducido], y_train)

proba_reducido = predecir_proba(modelo_reducido, X_test_enc[columnas_reducido])
pred_reducido = (proba_reducido >= 0.5).astype(int)
comparacion.append(evaluar_clasificador('Logística reducida (umbral 0.5)', y_test, pred_reducido))

razon_de_momios(modelo_reducido)

Logística reducida (umbral 0.5): exactitud 0.777 | precisión 0.754 | recall 0.623 | F1 0.683


,coef (log-odds),razón de momios,p-value
const,0.6426,1.901,0.0737
age,-0.0251,0.975,0.0140
es_menor,0.6358,1.889,0.1051
pclass_1,2.3746,10.747,0.0000
pclass_2,1.3341,3.797,0.0000
embarked_C,0.4968,1.643,0.0597
embarked_Q,0.8517,2.344,0.0229
sex_male,-2.5817,0.076,0.0000


---
# 8️⃣ Selección de variables: ¿y si lo hace el algoritmo? (15 min)

Acabas de quitar variables **a mano**, leyendo p-values. En la sesión de regresión lineal viste que **RFECV** hace eso solo: entrena, elimina la variable menos importante, repite, y usa validación cruzada para decidir con cuántas quedarse.

Antes de usarlo hay que resolver una pregunta que dejamos pendiente.

### ¿Hay que escalar en regresión logística?

**Para ajustar el modelo, no.** A diferencia de lo que verás en KNN, aquí el escalado no cambia las predicciones: si multiplicas una variable por 100, el coeficiente se divide entre 100 y la probabilidad estimada queda igual.

Pero hay tres situaciones en las que **sí** hace falta:

| Situación | ¿Escalar? | Por qué |
|---|---|---|
| Ajustar e interpretar coeficientes | ❌ No | En su escala original los coeficientes se leen en unidades reales: "por cada año de edad…" |
| **Comparar qué variable pesa más** | ✅ Sí | Un coeficiente pequeño puede serlo sólo porque la variable está en unidades grandes |
| **RFE / RFECV** | ✅ Sí | Eliminan por tamaño del coeficiente, así que sin escalar castigan a las variables de rango amplio |
| Usar `LogisticRegression` de sklearn | ✅ Sí | Aplica regularización L2 por defecto, y eso **sí** penaliza según la escala |

Compruébalo: el modelo escalado predice exactamente lo mismo, pero ordena los coeficientes distinto.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler()

X_train_escalado = pd.DataFrame(
    escalador.fit_transform(X_train_enc),
    columns=X_train_enc.columns, index=X_train_enc.index
)
X_test_escalado = pd.DataFrame(
    escalador.transform(X_test_enc),
    columns=X_test_enc.columns, index=X_test_enc.index
)

modelo_escalado = ajustar_logit(X_train_escalado, y_train)
proba_escalado = predecir_proba(modelo_escalado, X_test_escalado)

print("Diferencia máxima entre las probabilidades con y sin escalar:",
      round((proba_test - proba_escalado).abs().max(), 10))

pd.DataFrame({
    'coef sin escalar': modelo_completo.params.drop('const').abs().round(3),
    'coef escalado': modelo_escalado.params.drop('const').abs().round(3),
}).sort_values('coef escalado', ascending=False)

Diferencia máxima entre las probabilidades con y sin escalar: 0.0


,coef sin escalar,coef escalado
sex_male,2.618,1.253
pclass_1,2.341,1.000
pclass_2,1.336,0.545
age,0.025,0.325
embarked_Q,0.828,0.221
es_menor,0.708,0.218
embarked_C,0.498,0.197
viaja_solo,0.154,0.075
fare,0.001,0.039


✍️ **¿Qué observas?**
- ¿Cuánto cambian las probabilidades al escalar?
- Compara las dos columnas de coeficientes: ¿qué variable sube más de posición? ¿Por qué crees que pasa? (Pista: ¿en qué unidades está medida?)

- Las probabilidades **no cambian nada**: la diferencia máxima es 0.0.
- La que más sube de posición es **age**: sin escalar está casi hasta abajo (0.025) y escalada pasa al **4º lugar** (0.325). Pasa porque la edad se mide en años y va de 0 a 80, entonces el efecto de **un solo año** es pequeño, pero en todo su rango sí pesa bastante. Las dummies, en cambio, solo valen 0 o 1, así que su coeficiente ya estaba "en escala".

### Ahora sí: RFECV

Con los datos ya escalados, dejamos que el algoritmo elija. Usamos `StratifiedKFold` para que cada partición de la validación cruzada conserve la proporción de sobrevivientes, y `accuracy` como criterio.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

rfecv = RFECV(
    estimator=LogisticRegression(max_iter=1000),
    step=1,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy'
)
rfecv.fit(X_train_escalado, y_train)

columnas_rfecv = X_train_enc.columns[rfecv.support_]

print(f"Número óptimo de variables: {rfecv.n_features_}")
print(f"Variables seleccionadas: {list(columnas_rfecv)}")

Número óptimo de variables: 6
Variables seleccionadas: ['age', 'pclass_1', 'pclass_2', 'embarked_C', 'embarked_Q', 'sex_male']


In [ ]:
# Grafica la exactitud según el número de variables
# Llama a plot_rfecv_clasificacion(rfecv)
# ✍️ Tu código aquí
plot_rfecv_clasificacion(rfecv)
print(pd.Series(rfecv.cv_results_['mean_test_score'], index=rfecv.cv_results_['n_features']).round(4))

1    0.7894
2    0.7894
3    0.7838
4    0.7781
5    0.7866
6    0.8006
7    0.7978
8    0.7992
9    0.8006
dtype: float64


In [ ]:
# Ajusta el modelo con las variables que eligió RFECV, evalúalo y agrégalo a la comparación
# 1. modelo_rfecv = ajustar_logit(X_train_enc[columnas_rfecv], y_train)   ← sin escalar, para interpretarlo
# 2. proba_rfecv = predecir_proba(...)  y  pred_rfecv = (proba_rfecv >= 0.5).astype(int)
# 3. comparacion.append(evaluar_clasificador('Logística RFECV (umbral 0.5)', y_test, pred_rfecv))
# ✍️ Tu código aquí
modelo_rfecv = ajustar_logit(X_train_enc[columnas_rfecv], y_train)
proba_rfecv = predecir_proba(modelo_rfecv, X_test_enc[columnas_rfecv])
pred_rfecv = (proba_rfecv >= 0.5).astype(int)
comparacion.append(evaluar_clasificador('Logística RFECV (umbral 0.5)', y_test, pred_rfecv))

print("Descartadas por RFECV:", list(X_train_enc.columns.drop(columnas_rfecv)))
razon_de_momios(modelo_rfecv)

Logística RFECV (umbral 0.5): exactitud 0.771 | precisión 0.719 | recall 0.667 | F1 0.692
Descartadas por RFECV: ['fare', 'viaja_solo', 'es_menor']


,coef (log-odds),razón de momios,p-value
const,1.0144,2.758,0.0003
age,-0.0352,0.965,0.0000
pclass_1,2.3990,11.012,0.0000
pclass_2,1.3363,3.805,0.0000
embarked_C,0.5024,1.653,0.0560
embarked_Q,0.8372,2.310,0.0255
sex_male,-2.5903,0.075,0.0000


✍️ **Compara los dos caminos:**
- ¿Cuántas variables eligió RFECV? ¿Cuáles descartó?
- ¿Coinciden con las que tú habías quitado por p-value? ¿Descartó alguna más?
- Mira la curva: ¿a partir de cuántas variables deja de mejorar? ¿Qué criterio crees que usa RFECV cuando hay empate?

- RFECV eligió **6 variables**: age, pclass_1, pclass_2, embarked_C, embarked_Q y sex_male. Descartó **fare, viaja_solo y es_menor**.
- Sí coincide con las dos que yo quité por p-value (fare y viaja_solo), y además descartó **es_menor**, que tampoco era significativa (p = 0.084).
- En la curva, con 1 o 2 variables ya llega a 0.789. Luego baja un poco, sube hasta **0.8006 con 6 variables** y ahí se queda casi igual: con 9 variables también da 0.8006. Creo que cuando hay empate RFECV se queda con el **menor número de variables**, o sea el modelo más simple.

---
# 9️⃣ Conclusiones (10 min)

In [ ]:
# Tabla comparativa de todo lo que probamos hoy
# Pista: pd.DataFrame(comparacion)
# ✍️ Tu código aquí
pd.DataFrame(comparacion)

,modelo,exactitud,precisión,recall,F1
0,Logística (umbral 0.5),0.771,0.741,0.623,0.677
1,Todos mueren,0.615,0.000,0.000,0.000
2,Regla: sobreviven mujeres,0.777,0.738,0.652,0.692
3,Logística (umbral 0.4),0.782,0.697,0.768,0.731
4,Logística reducida (umbral 0.5),0.777,0.754,0.623,0.683
5,Logística RFECV (umbral 0.5),0.771,0.719,0.667,0.692


✍️ **Responde la pregunta del inicio:** ¿quién sobrevivió al Titanic? Usa tus coeficientes y momios para sustentarlo.

Sobrevivieron más que nada las **mujeres**, la gente de **primera y segunda clase** y los **más jóvenes**. Lo veo en los momios del modelo:
- Si eras **hombre**, tus momios de sobrevivir se multiplicaban por **0.075**, o sea casi no tenías chance comparado con una mujer.
- Si ibas en **primera clase**, tus momios eran como **11 veces** más altos que en tercera. En **segunda clase**, como **3.8 veces**.
- Entre **más edad**, menos chance: cada año que tenías de más bajaba un poquito la probabilidad de sobrevivir.

El puerto en realidad no importa, solo refleja en qué clase viajaba la gente. El modelo le atina a como **77 de cada 100** pasajeros que no había visto, y si bajo el umbral a 0.4 sube a como **78 de cada 100**.

## 📝 Lo que aprendimos
✍️ Escribe 3 cosas que aprendiste hoy.

1. Que para predecir algo de sí o no se usa la regresión logística, porque da una probabilidad entre 0 y 1.
2. Que primero se separan los datos en train y test y luego se rellenan los nulos, para no hacer trampa con los datos de prueba.
3. Que la exactitud sola puede engañar: decir que "todos mueren" le atina a 61.5% pero no encuentra a ningún sobreviviente. Por eso hay que ver también la matriz de confusión, la precisión y el recall.

## 🚀 Retos opcionales
- Con `plot_metricas_por_umbral` busca el umbral que maximiza el F1 y evalúa ese modelo. ¿Le gana al de umbral 0.4?
- Agrega `adult_male` al modelo (esa que quitamos al inicio). ¿Qué le pasa a los p-values de `sex_male` y `es_menor`? ¿Por qué?
- Corre `RFE` con `n_features_to_select=4` sobre los datos escalados. ¿Qué variables elige y qué exactitud logra?
- Ajusta la logística usando **sólo** `sex_male`. ¿Qué exactitud da? Compárala con la regla "las mujeres sobreviven" y explica el resultado